In [ ]:
# import os
# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()

True

In [ ]:
# client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com/v1")

In [ ]:
# SYSTEM_PROMPT = """
# You must format every response using ONLY these tags:

# - <*&TEXT&*> for text
# - <*&CODE:filename=...:lang=...:downloadable=true|false&*> for code
# - <*&COMMAND&*> for commands
# - <*&IMAGE:path=...&*> for images
# - <*&PDF:path=...&*> for PDF files
# - <*&AUDIO:path=...&*> for audio
# - <*&YOUTUBE:url=...&*> for YouTube embeds
# - <*&END&*> to end the response (required)

# Rules:
# - ALWAYS end with <*&END&*>
# - DO NOT output anything outside tags
# - Tags are case-sensitive and must match exactly
# - You can use multiple TEXT, CODE, etc. blocks in one response
# - Maintain correct order (TEXT → CODE → TEXT → COMMAND → ...)

# CODE rules:
# - downloadable=true ONLY if the code is a complete runnable file
# - otherwise downloadable=false

# Formatting rules:
# - No spaces inside tag brackets
# - All attributes must be included exactly as shown
# - Do not invent new tags

# Examples:

# <*&TEXT&*>Here is your code:
# <*&CODE:filename=main.py:lang=python:downloadable=true&*>print("Hello")
# <*&TEXT&*>Run it:
# <*&COMMAND&*>python3 main.py
# <*&END&*>

# <*&TEXT&*>Watch this:
# <*&YOUTUBE:url=https://www.youtube.com/embed/CG48pSyK8GU&*>
# <*&END&*>
# """

In [ ]:
# for r in client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user", "content": "write hello world python example and how to run it"}
#     ],
#     stream=True
# ):
#     print(r.choices[0].delta.content, end="")

<*&TEXT&*>Here's a simple "Hello, World!" Python script:

<*&CODE:filename=hello.py:lang=python:downloadable=true&*>print("Hello, World!")
<*&TEXT&*>To run this script, use this command in your terminal:

<*&COMMAND&*>python3 hello.py
<*&END&*>

In [3]:
from tools.PythonRunner import PythonRunner


In [7]:
print(PythonRunner.createNewRunner.args_schema.model_json_schema())

{'description': 'Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.', 'properties': {'details': {'description': "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').", 'title': 'Details', 'type': 'string'}}, 'required': ['details'], 'title': 'CreateNewRunner', 'type': 'object'}


In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool
import json

In [9]:
convert_to_openai_tool(PythonRunner.createNewRunner)

{'type': 'function',
 'function': {'name': 'createNewRunner',
  'description': 'Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.',
  'parameters': {'properties': {'details': {'description': "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').",
     'type': 'string'}},
   'required': ['details'],
   'type': 'object'}}}

In [17]:
print(convert_to_openai_tool(PythonRunner.createNewRunner))

{'type': 'function', 'function': {'name': 'createNewRunner', 'description': 'Use when and if you need to create a python code runner process. The runner stays open to recieve code and execute it untill you end the process. Only the runner is in sandbox but the rest of agent is out.', 'parameters': {'properties': {'details': {'description': "A description you want to attach to the runner to remember what the purpose of this runner is and its details (ex. 'Create a machine learning model to detect houses pricing based on input data. the code will read '/home/user/datasets/houses.csv' file then do some preprocessing and cleaning then scaling then train KNN model then save it in '.pkl' file.').", 'type': 'string'}}, 'required': ['details'], 'type': 'object'}}}


In [15]:
from tools import time_utils

In [16]:
print(json.dumps(convert_to_openai_tool(time_utils.time), indent=2))

{
  "type": "function",
  "function": {
    "name": "time",
    "description": "Returns the current time as a string in formate \"%Y-%m-%d %H:%M:%S\". Used when current date and time are required. If you output to user, use a letteral format to be readable like: \"It's three oclock PM on fifth of September\" instead of numbers to be pronounced well.",
    "parameters": {
      "properties": {},
      "type": "object"
    }
  }
}


In [1]:
import socketio

In [2]:
sio = socketio.Client()

In [22]:
@sio.event
def connect():
    print("✅ Connected to server")

    # join chat
    sio.emit("join_chat", {
        "chat_id": 1
    })

    # send test message
    sio.emit("send_message", {
        "chat_id": 1,
        "content": "Write me Python hello world code"
    })


@sio.event
def disconnect():
    print("\n❌ Disconnected from server")


# =========================
# STREAM EVENTS
# =========================

@sio.on("stream_start")
def on_stream_start(data):

    print("\n==============================")
    print("🤖 Assistant started streaming")
    print("==============================\n")


@sio.on("stream_chunk")
def on_stream_chunk(data):

    chunk = data.get("chunk", "")

    # print chunk instantly
    print(chunk, end="", flush=True)


@sio.on("stream_end")
def on_stream_end(data):

    print("\n\n==============================")
    print("✅ Stream finished")
    print("==============================\n")


@sio.on("stream_error")
def on_stream_error(data):

    print("\n❌ STREAM ERROR")
    print(data)


@sio.on("message_sent")
def on_message_sent(data):

    print("📨 Message sent:", data)


@sio.on("error")
def on_error(data):

    print("❌ ERROR:", data)

In [23]:
sio.connect("http://localhost:5000")

ConnectionError: Already connected

In [24]:
connect()

✅ Connected to server


In [25]:
sio.emit("send_message", {
        "chat_id": 1,
        "content": "Hello agent, write Python code"
    })

sio.wait()

KeyboardInterrupt: 